In [1]:
from pathlib import Path

import pandas as pd

In [2]:
PROCESSED_DATA_DIR = Path("../data/processed")

CLEANED_TITLES_PATH = (
    PROCESSED_DATA_DIR / "streaming_titles_clean.csv"
)

In [3]:
streaming_titles = pd.read_csv(
    CLEANED_TITLES_PATH,
    parse_dates=["date_added"]
)

print("Dataset shape:", streaming_titles.shape)
print("Date type:", streaming_titles["date_added"].dtype)
print("Duplicate content IDs:", streaming_titles["content_id"].duplicated().sum())

Dataset shape: (10257, 16)
Date type: datetime64[us]
Duplicate content IDs: 0


In [7]:
COUNTRY_COLUMNS = [
    "content_id",
    "platform",
    "type",
    "title",
    "country",
    "release_year",
    "date_added",
    "rating"
]

country_analysis = streaming_titles[COUNTRY_COLUMNS].copy()

country_analysis["country"] = (
    country_analysis["country"]
    .str.split(",")
)

country_analysis = country_analysis.explode(
    "country",
    ignore_index=True
)

country_analysis["country"] = (
    country_analysis["country"]
    .astype("string")
    .str.strip()
)

country_analysis["country"] = (
    country_analysis["country"]
    .replace("", "Unknown")
    .fillna("Unknown")
)

country_analysis = country_analysis.drop_duplicates(
    subset=["content_id", "country"]
).reset_index(drop=True)

country_analysis["country_count"] = (
    country_analysis
    .groupby("content_id")["country"]
    .transform("count")
)

country_analysis["title_credit"] = 1

country_analysis["fractional_credit"] = (
    1 / country_analysis["country_count"]
)

In [8]:
print("Country table rows:", len(country_analysis))

print(
    "Unique titles represented:",
    country_analysis["content_id"].nunique()
)

print(
    "Duplicate title-country relationships:",
    country_analysis[
        ["content_id", "country"]
    ].duplicated().sum()
)

print(
    "Missing countries:",
    country_analysis["country"].isna().sum()
)

Country table rows: 12597
Unique titles represented: 10257
Duplicate title-country relationships: 0
Missing countries: 0


In [9]:
country_credit_check = (
    country_analysis
    .groupby("platform")["fractional_credit"]
    .sum()
    .round(6)
)

display(country_credit_check)

platform
Disney+    1450.0
Netflix    8807.0
Name: fractional_credit, dtype: float64

In [10]:
top_countries_full = (
    country_analysis[
        country_analysis["country"] != "Unknown"
    ]
    .groupby(["platform", "country"])
    .agg(
        title_associations=("title_credit", "sum"),
        unique_titles=("content_id", "nunique")
    )
    .reset_index()
    .sort_values(
        ["platform", "title_associations"],
        ascending=[True, False]
    )
)

display(
    top_countries_full
    .groupby("platform")
    .head(10)
)

,platform,country,title_associations,unique_titles
46,Disney+,United States,1184,1184
45,Disney+,United Kingdom,101,101
7,Disney+,Canada,77,77
2,Disney+,Australia,23,23
12,Disney+,France,22,22
36,Disney+,South Korea,13,13
8,Disney+,China,10,10
20,Disney+,Japan,10,10
13,Disney+,Germany,9,9
19,Disney+,Ireland,8,8


In [11]:
top_countries_fractional = (
    country_analysis[
        country_analysis["country"] != "Unknown"
    ]
    .groupby(["platform", "country"])
    .agg(
        fractional_titles=("fractional_credit", "sum")
    )
    .reset_index()
    .sort_values(
        ["platform", "fractional_titles"],
        ascending=[True, False]
    )
)

display(
    top_countries_fractional
    .groupby("platform")
    .head(10)
)

,platform,country,fractional_titles
46,Disney+,United States,1082.399242
45,Disney+,United Kingdom,54.182576
7,Disney+,Canada,36.065909
12,Disney+,France,9.357576
2,Disney+,Australia,9.174242
13,Disney+,Germany,4.083333
36,Disney+,South Korea,3.774242
8,Disney+,China,3.324242
20,Disney+,Japan,3.324242
17,Disney+,India,2.900000


In [12]:
GENRE_COLUMNS = [
    "content_id",
    "platform",
    "type",
    "title",
    "listed_in",
    "release_year",
    "date_added",
    "rating"
]

genre_analysis = streaming_titles[GENRE_COLUMNS].copy()

In [13]:
genre_analysis["genre"] = (
    genre_analysis["listed_in"]
    .str.split(",")
)

genre_analysis = genre_analysis.explode(
    "genre",
    ignore_index=True
)

genre_analysis["genre"] = (
    genre_analysis["genre"]
    .astype("string")
    .str.strip()
)

In [14]:
genre_analysis["genre"] = (
    genre_analysis["genre"]
    .replace("", "Unknown")
    .fillna("Unknown")
)

genre_analysis = genre_analysis.drop_duplicates(
    subset=["content_id", "genre"]
).reset_index(drop=True)

genre_analysis = genre_analysis.drop(
    columns="listed_in"
)

In [16]:
genre_analysis["genre_count"] = (
    genre_analysis
    .groupby("content_id")["genre"]
    .transform("count")
)

genre_analysis["title_credit"] = 1

genre_analysis["fractional_credit"] = (
    1 / genre_analysis["genre_count"]
)

print("Genre table rows:", len(genre_analysis))

print(
    "Unique titles represented:",
    genre_analysis["content_id"].nunique()
)

print(
    "Duplicate title-genre relationships:",
    genre_analysis[
        ["content_id", "genre"]
    ].duplicated().sum()
)

print(
    "Missing genres:",
    genre_analysis["genre"].isna().sum()
)

genre_credit_check = (
    genre_analysis
    .groupby("platform")["fractional_credit"]
    .sum()
    .round(6)
)

display(genre_credit_check)

Genre table rows: 23233
Unique titles represented: 10257
Duplicate title-genre relationships: 0
Missing genres: 0


platform
Disney+    1450.0
Netflix    8807.0
Name: fractional_credit, dtype: float64

In [17]:
genre_summary = (
    genre_analysis
    .groupby(["platform", "genre"])
    .agg(
        unique_titles=("content_id", "nunique"),
        fractional_titles=("fractional_credit", "sum")
    )
    .reset_index()
    .sort_values(
        ["platform", "unique_titles"],
        ascending=[True, False]
    )
)

display(
    genre_summary
    .groupby("platform")
    .head(15)
)

,platform,genre,unique_titles,fractional_titles
16,Disney+,Family,632,223.833333
2,Disney+,Animation,542,191.333333
7,Disney+,Comedy,526,190.166667
0,Disney+,Action-Adventure,452,156.666667
1,Disney+,Animals & Nature,208,78.166667
8,Disney+,Coming of Age,205,72.666667
17,Disney+,Fantasy,192,70.833333
13,Disney+,Documentary,174,88.166667
20,Disney+,Kids,141,51.500000
15,Disney+,Drama,134,50.333333


In [18]:
COUNTRY_ANALYSIS_PATH = (
    PROCESSED_DATA_DIR / "streaming_titles_by_country.csv"
)

GENRE_ANALYSIS_PATH = (
    PROCESSED_DATA_DIR / "streaming_titles_by_genre.csv"
)

In [19]:
country_analysis.to_csv(
    COUNTRY_ANALYSIS_PATH,
    index=False
)

genre_analysis.to_csv(
    GENRE_ANALYSIS_PATH,
    index=False
)

In [20]:
print(
    "Country table created:",
    COUNTRY_ANALYSIS_PATH.exists()
)

print(
    "Genre table created:",
    GENRE_ANALYSIS_PATH.exists()
)

Country table created: True
Genre table created: True


## Analysis Table Results

- The cleaned title-level dataset remains unchanged at 10,257 records.
- A separate country table was created with one row per title-country relationship.
- A separate genre table was created with one row per title-genre relationship.
- Every title remains represented in both analysis tables.
- Duplicate title-country and title-genre relationships were removed.
- Full-count measures were created to measure category participation.
- Fractional-count measures were created to prevent multi-country or multi-genre titles from contributing more than one total title.
- `Unknown` country values were preserved but can be excluded from country rankings.
- The country and genre tables were exported to the processed-data folder.
- Genre terminology will require careful review before direct comparisons are made between platforms.